In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
import numpy as np
import os
import shutil

In [ ]:
def mild_lowlight_preprocess(image):
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    image = tf.image.adjust_gamma(image, 1.3)
    image = image * (1.0 - 0.1)
    image = image + tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=0.03)
    image = tf.clip_by_value(image, -255.0, 255.0)
    return image

In [ ]:
batch_size = 32
img_size = (224, 224)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
drive_validation_dir = "/content/drive/MyDrive/Emotions Dataset/test"
local_validation_dir = "/content/validation"

In [ ]:
if not os.path.exists(local_validation_dir):
    print("Copying validation data to local storage")
    shutil.copytree(drive_validation_dir, local_validation_dir)
    print("Validation data copied successfully")

In [ ]:
# Generator: Low Light Data
test_datagen = ImageDataGenerator(preprocessing_function=mild_lowlight_preprocess)
test_generator = test_datagen.flow_from_directory(
    local_validation_dir, target_size=img_size, batch_size=batch_size, class_mode='categorical', shuffle=True
)

Found 2278 images belonging to 3 classes.


In [ ]:
# Load model
print("Loading Baseline Model")
model = load_model('model_A_baseline_no_augmentation.h5')

# Evaluate
print("Evaluating Baseline on Low-Light Data")
# The model returns [loss, accuracy, top_2_accuracy]
results = model.evaluate(test_generator, verbose=1)

loss_pre = results[0]
acc_pre = results[1]
# results[2] is top_2_accuracy, which we can ignore for now

print(f"Baseline Accuracy: {acc_pre:.2%}")

# 3. CONFIGURE AdaBN
print("\nConfiguring Adaptive Batch Normalization (AdaBN)")

# Freeze the entire model first
model.trainable = False

# We find the inner EfficientNet and Unfreeze only Batch Normalization layers

efficientnet_layer = model.get_layer('efficientnetb0')
efficientnet_layer.trainable = True

# Freeze everything inside EfficientNet exept BatchNormalization
for layer in efficientnet_layer.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True
    else:
        layer.trainable = False

# 4. recompile
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("AdaBN Configuration Complete. Ready for adaptation.")

Loading Baseline Model


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Evaluating Baseline on Low-Light Data
72/72 ━━━━━━━━━━━━━━━━━━━━ 38s 279ms/step - accuracy: 0.5727 - loss: 1.2452 - top_2_accuracy: 0.8546
Baseline Accuracy: 58.38%

Configuring Adaptive Batch Normalization (AdaBN)
AdaBN Configuration Complete. Ready for adaptation.


In [ ]:
# Adaptation Run
print("Adapting to Low-Light Domain (1 Epoch)")
model.fit(test_generator, epochs=1)

Adapting to Low-Light Domain (1 Epoch)


/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/trainer.py:83: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


72/72 ━━━━━━━━━━━━━━━━━━━━ 40s 309ms/step - accuracy: 0.3786 - loss: 5.3484


In [ ]:
# Final Evaluation
print("\nEvaluating after AdaBN")
loss_post, acc_post = model.evaluate(test_generator, verbose=1)
print(f"AdaBN Accuracy: {acc_post:.2%}")
print(f"Improvement: +{acc_post - acc_pre:.2%}")


Evaluating after AdaBN
72/72 ━━━━━━━━━━━━━━━━━━━━ 22s 215ms/step - accuracy: 0.4145 - loss: 2.3965
AdaBN Accuracy: 42.80%
Improvement: +-15.58%
